# Machine-checked reproduction and repeatability

This notebook derives every number in the paper's Section 6 from the raw
artifacts committed under `docs/paper/artifacts/`. Nothing here needs the
Pi: the transcripts and campaign outputs are parsed as-is. The last cell
shows how to rerun everything live against a Pi serving the FSM.

In [1]:
import json, statistics
from pathlib import Path

ART = Path("../docs/paper/artifacts")
transcript = ART / "mcp_flood2_recreation_2026-07-11.jsonl"
events = [json.loads(l) for l in transcript.read_text().splitlines() if l.startswith("{")]
[e.get("action") for e in events if "action" in e]

['characterize',
 'baseline',
 'hypothesize',
 'implement',
 'compile_',
 'verify',
 'benchmark',
 'evaluate',
 'log_variant',
 'hypothesize',
 'implement',
 'compile_',
 'verify',
 'benchmark',
 'log_variant']

## The re-derivation

The FSM measured its own baseline, accepted the fused strip-2 kernel as
experiment 1, gated it, benchmarked it, and issued its own verdict.

In [2]:
for e in events:
    r = e.get("result", {})
    inner = r.get("result", {}) if isinstance(r, dict) else {}
    if e.get("action") == "baseline":
        print("baseline:", inner)
    if e.get("action") == "log_variant" and isinstance(inner.get("logged"), dict):
        print("ledger:  ", json.dumps(inner["logged"]))

baseline: {'baseline_steps_per_sec': 1346.8013468013469, 'gate_ok': True}
ledger:   {"exp": 1, "fused": true, "strip": 2, "compile_ok": true, "verify_ok": true, "steps_per_sec": 2116.4021164021165, "best_sps": 2116.4021164021165, "verdict": "keep"}
ledger:   {"exp": 2, "fused": true, "strip": 2, "compile_ok": true, "verify_ok": false, "steps_per_sec": null, "best_sps": 2116.4021164021165, "verdict": "revert"}


## The gate refusal

The driver submits a mass-violating kernel (rainfall doubled in one of the
two cell updates), then requests `benchmark` after the gate fails. The
server's verbatim answer:

In [3]:
for e in events:
    if e.get("action") == "benchmark" and isinstance(e.get("result"), dict) \
            and e["result"].get("error") == "invalid_transition":
        print(json.dumps(e["result"], indent=1))

{
 "error": "invalid_transition",
 "requested": "benchmark",
 "valid_next_actions": [
  "log_variant"
 ],
 "next_action_schemas": {
  "log_variant": {}
 },
 "message": "action 'benchmark' is not reachable from current state. Valid actions now: ['log_variant'].",
 "next_hint": "Action 'benchmark' is not reachable from the current state. Reachable now: log_variant."
}


## Repeatability: pass^k

`passk_flood2.py` runs the whole two-cycle reproduction k times from cool
starts and scores each run. The 2026-08-09 campaign:

In [4]:
summary = json.loads((ART / "passk_2026-08-09" / "summary.json").read_text())
for r in summary["runs"]:
    print(f"run {r['run']}: pass={r['pass']}  baseline={r['baseline']:.1f}  "
          f"kept={r['kept']:.1f}  speedup={r['kept']/r['baseline']:.3f}")
print()
print(summary["summary"])
assert summary["summary"]["pass_k"], "pass^k failed"
print("\npass^{} = {}/{}".format(summary["summary"]["k"], summary["summary"]["passes"], summary["summary"]["k"]))

run 1: pass=True  baseline=1351.4  kept=2127.7  speedup=1.574
run 2: pass=True  baseline=1342.3  kept=2127.7  speedup=1.585
run 3: pass=True  baseline=1351.4  kept=2127.7  speedup=1.574
run 4: pass=True  baseline=1351.4  kept=2127.7  speedup=1.574
run 5: pass=True  baseline=1346.8  kept=2127.7  speedup=1.580

{'k': 5, 'passes': 5, 'pass_k': True, 'baseline_mean': 1348.6, 'baseline_stdev': 4.1, 'kept_mean': 2127.7, 'kept_stdev': 0.0, 'speedups': [1.574, 1.585, 1.574, 1.574, 1.58]}

pass^5 = 5/5


## The agent-driven session

One session (2026-08-02) was driven end to end by a language model over
MCP with a three-experiment budget. Its ledger entries, extracted from the
tool-call transcript:

In [5]:
import re
sess = ART / "claude_sessions" / "20260802T150755Z"
raw = (sess / "ledger_responses.json").read_text()
seen = set()
for m in re.finditer(r'\\?"logged\\?":\s*\{(.*?)\}', raw.replace('\\"', '"')):
    entry = "{" + m.group(1) + "}"
    if entry not in seen:
        seen.add(entry)
        print(entry)

{"exp":1,"fused":false,"strip":4,"compile_ok":true,"verify_ok":true,"steps_per_sec":1470.5882352941176,"best_sps":1470.5882352941176,"verdict":"keep"}
{"exp":2,"fused":false,"strip":8,"compile_ok":true,"verify_ok":true,"steps_per_sec":921.6589861751152,"best_sps":1470.5882352941176,"verdict":"revert"}
{"exp":3,"fused":false,"strip":6,"compile_ok":true,"verify_ok":true,"steps_per_sec":1298.7012987012988,"best_sps":1470.5882352941176,"verdict":"revert"}


## Rerunning live

On the Pi: `cd /root/seppa && .venv/bin/python theodosia_server.py --http --flood2`

From any machine on the network:

```
.venv/bin/python drive_flood2_mcp.py http://<pi>:8000/mcp   # one reproduction
python3 passk_flood2.py 5 http://<pi>:8000/mcp              # scored campaign
```